CONFIGURAÇÕES

In [0]:
# Configurações
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
VOLUME = "cinedata_inputs"

INPUT_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"

print(INPUT_PATH)

/Volumes/workspace/bronze/cinedata_inputs


In [0]:
# Imports
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp

In [0]:
# Mapeamento dos arquivos
BRONZE_TABLES = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

INGESTION DOS CSV's NA BRONZE

In [0]:
def ingest_csv_to_bronze(file_name: str, table_name: str) -> None:
    """
    Realiza a ingestão de um CSV para a camada Bronze.

    Durante o desenvolvimento, evita inserir novamente
    o mesmo arquivo quando a tabela já contém os registros
    esperados.
    """

    file_path = f"{INPUT_PATH}/{file_name}"
    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    print(f"\n{'=' * 60}")
    print(f"Arquivo: {file_name}")
    print(f"Tabela:  {full_table_name}")

    # 1. Leitura do arquivo original
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )

    expected_count = df.count()

    print(f"Registros no CSV: {expected_count}")

    # 2. Verifica se a tabela já existe
    if spark.catalog.tableExists(full_table_name):

        current_count = spark.table(full_table_name).count()

        print(f"Registros na Bronze: {current_count}")

        # Quantidade exatamente igual:
        # provavelmente este arquivo já foi ingerido.
        if current_count == expected_count:
            print(
                "[IGNORADO] A tabela já contém a quantidade "
                "esperada de registros."
            )
            return

        # Quantidade maior:
        # provavelmente ocorreu reprocessamento durante os testes.
        elif current_count > expected_count:
            print(
                "[ATENÇÃO] A tabela possui mais registros "
                "do que o arquivo de origem."
            )

            print(
                f"Esperado: {expected_count} | "
                f"Encontrado: {current_count}"
            )

            print("Recriando a tabela Bronze...")

            spark.sql(
                f"DROP TABLE IF EXISTS {full_table_name}"
            )

        else:
            # Se estiver menor, também recriamos.
            # Isso evita deixar uma ingestão parcial.
            print(
                "[ATENÇÃO] A tabela possui menos registros "
                "do que o arquivo de origem."
            )

            print("Recriando a tabela Bronze...")

            spark.sql(
                f"DROP TABLE IF EXISTS {full_table_name}"
            )

    # 3. Adiciona timestamp da ingestão
    df_bronze = (
        df.withColumn(
            "ingestion_datetime",
            current_timestamp()
        )
    )

    # 4. Persistência em Delta
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(full_table_name)
    )

    print(
        f"[OK] Ingestão concluída: "
        f"{full_table_name}"
    )

In [0]:
# Execução da ingestion
for file_name, table_name in BRONZE_TABLES.items():
    ingest_csv_to_bronze(
        file_name=file_name,
        table_name=table_name
    )


Arquivo: movies_info_TMDB_IMDB.csv
Tabela:  workspace.bronze.tb_movies_info
Registros no CSV: 106930
Registros na Bronze: 213860
[ATENÇÃO] A tabela possui mais registros do que o arquivo de origem.
Esperado: 106930 | Encontrado: 213860
Recriando a tabela Bronze...
[OK] Ingestão concluída: workspace.bronze.tb_movies_info

Arquivo: movies_financials_IMDB_TMDB.csv
Tabela:  workspace.bronze.tb_movies_financials
Registros no CSV: 106165
Registros na Bronze: 106165
[IGNORADO] A tabela já contém a quantidade esperada de registros.

Arquivo: movies_metrics_IMDB_TMDB.csv
Tabela:  workspace.bronze.tb_movies_metrics
Registros no CSV: 107364
Registros na Bronze: 107364
[IGNORADO] A tabela já contém a quantidade esperada de registros.

Arquivo: credits_and_tags_IMDB_TMDB.csv
Tabela:  workspace.bronze.tb_credits_and_tags
Registros no CSV: 106320
Registros na Bronze: 106320
[IGNORADO] A tabela já contém a quantidade esperada de registros.

Arquivo: movies_reviews.csv
Tabela:  workspace.bronze.tb_mov

VALIDAÇÃO DA INGESTION DOS CSV's

In [0]:

# VALIDAÇÃO DA CAMADA BRONZE"

for file_name, table_name in BRONZE_TABLES.items():

    file_path = f"{INPUT_PATH}/{file_name}"
    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    df_source = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )

    source_count = df_source.count()

    if not spark.catalog.tableExists(full_table_name):
        print(
            f"[ERRO] {table_name:<30} "
            f"Tabela não encontrada."
        )
        continue

    bronze_count = spark.table(full_table_name).count()

    if bronze_count == source_count:
        status = "OK"
    elif bronze_count > source_count:
        status = "EXCESSO"
    else:
        status = "FALTANDO"

    print(
        f"[{status:<7}] "
        f"{table_name:<30} "
        f"Origem: {source_count:<10} "
        f"Bronze: {bronze_count:<10}"
    )

[OK     ] tb_movies_info                 Origem: 106930     Bronze: 106930    
[OK     ] tb_movies_financials           Origem: 106165     Bronze: 106165    
[OK     ] tb_movies_metrics              Origem: 107364     Bronze: 107364    
[OK     ] tb_credits_and_tags            Origem: 106320     Bronze: 106320    
[OK     ] tb_movies_reviews              Origem: 32412      Bronze: 32412     


In [0]:
# Validação da ingestion_datetime
for table_name in BRONZE_TABLES.values():

    full_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"

    df = spark.table(full_table_name)

    if "ingestion_datetime" in df.columns:
        print(f"[OK] {table_name}: ingestion_datetime encontrada.")
    else:
        print(f"[ERRO] {table_name}: ingestion_datetime não encontrada.")

[OK] tb_movies_info: ingestion_datetime encontrada.
[OK] tb_movies_financials: ingestion_datetime encontrada.
[OK] tb_movies_metrics: ingestion_datetime encontrada.
[OK] tb_credits_and_tags: ingestion_datetime encontrada.
[OK] tb_movies_reviews: ingestion_datetime encontrada.


In [0]:
# visualização
display(
    spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.tb_movies_info"
    )
)

id,tconst,title,original_title,original_language,release_date,runtime,status,overview,tagline,ingestion_datetime
293660,tt1431045,Deadpool,Deadpool,en,2016-02-09,108,Released,"The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.",Witness the beginning of a happy ending.,2026-09-19T20:10:34.961Z
299536,tt4154756,AVENGERS: INFINITY WAR,Avengers: Infinity War,en,04-25-2018,149,Released,"As the Avengers and their allies have continued to protect the world from threats too large for any one hero to handle, a new danger has emerged from the cosmic shadows: Thanos. A despot of intergalactic infamy, his goal is to collect all six Infinity Stones, artifacts of unimaginable power, and use them to inflict his twisted will on all of reality. Everything the Avengers have fought for has led up to this moment - the fate of Earth and existence itself has never been more uncertain.",An entire universe. Once and for all.,2026-09-19T20:10:34.961Z
299534,tt4154796,Avengers: Endgame,Avengers: Endgame,en,2019-04-24,181,released,"After the devastating events of Avengers: Infinity War, the universe is in ruins due to the efforts of the Mad Titan, Thanos. With the help of remaining allies, the Avengers must assemble once more in order to undo Thanos' actions and restore order to the universe once and for all, no matter what consequences may be in store.",Avenge the fallen.,2026-09-19T20:10:34.961Z
475557,tt7286456,Joker,Joker,en,2019-10-01,122,Released,"During the 1980s, a failed stand-up comedian is driven insane and turns to a life of crime and chaos in Gotham City while becoming an infamous psychopathic crime figure.",Put on a happy face.,2026-09-19T20:10:34.961Z
271110,tt3498820,Captain America: Civil War,Captain America: Civil War,en,2016-04-27,147,Released,"Following the events of Age of Ultron, the collective governments of the world pass an act designed to regulate all superhuman activity. This polarizes opinion amongst the Avengers, causing two factions to side with Iron Man or Captain America, which causes an epic battle between former allies.",United we stand. Divided we fall.,2026-09-19T20:10:34.961Z
284054,tt1825683,Black Panther,Black Panther,en,2018-02-13,135,Released,"King T'Challa returns home to the reclusive, technologically advanced African nation of Wakanda to serve as his country's new leader. However, T'Challa soon finds that he is challenged for the throne by factions within his own country as well as without. Using powers reserved to Wakandan kings, T'Challa assumes the Black Panther mantle to join with ex-girlfriend Nakia, the queen-mother, his princess-kid sister, members of the Dora Milaje (the Wakandan 'special forces') and an American secret agent, to prevent Wakanda from being dragged into a world war.",null,2026-09-19T20:10:34.961Z
284052,tt1211837,Doctor Strange,Doctor Strange,en,2016-10-25,115,Released,"After his career is destroyed, a brilliant but arrogant surgeon gets a new lease on life when a sorcerer takes him under her wing and trains him to defend the world against evil.",The impossibilities are endless.,2026-09-19T20:10:34.961Z
315635,tt2250912,Spider-Man: Homecoming,Spider-Man: Homecoming,en,2017-07-05,133,RELEASED,"Following the events of Captain America: Civil War, Peter Parker, with the help of his mentor Tony Stark, tries to balance his life as an ordinary high school student in Queens, New York City, with fighting crime as his superhero alter ego Spider-Man as a new threat, the Vulture, emerges.",Homework can wait. The city can't.,2026-09-19T20:10:34.961Z
283995,tt3896198,Guardians of the Galaxy Vol. 2,Guardians of the Galaxy Vol. 2,en,2017-04-19,137,Released,The Guardians must fight to keep their newfound family together as t

INGESTION DA COTAÇÃO DO BANCO CENTRAL

In [0]:
from datetime import datetime, timedelta

# Datas padrão: últimos 7 dias corridos
data_fim_default = datetime.now()
data_inicio_default = data_fim_default - timedelta(days=7)

data_inicio_default = data_inicio_default.strftime("%m-%d-%Y")
data_fim_default = data_fim_default.strftime("%m-%d-%Y")

# Remove widgets anteriores caso esta célula seja reexecutada
dbutils.widgets.removeAll()

# Criação dos parâmetros
dbutils.widgets.text(
    "data_inicio",
    data_inicio_default,
    "Data Inicial (MM-DD-AAAA)"
)

dbutils.widgets.text(
    "data_fim",
    data_fim_default,
    "Data Final (MM-DD-AAAA)"
)

In [0]:
# Recuperação dos widgets
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Data inicial: {data_inicio}")
print(f"Data final:   {data_fim}")

# Validação do formato MM-DD-AAAA
try:
    inicio = datetime.strptime(data_inicio, "%m-%d-%Y")
    fim = datetime.strptime(data_fim, "%m-%d-%Y")

except ValueError:
    raise ValueError(
        "As datas devem estar no formato MM-DD-AAAA. "
        "Exemplo: 09-19-2026"
    )

if inicio > fim:
    raise ValueError(
        "A data inicial não pode ser posterior à data final."
    )

print("[OK] Período informado é válido.")

Data inicial: 09-12-2026
Data final:   09-19-2026
[OK] Período informado é válido.


In [0]:
# Construção da URL da API do Banco Central
BCB_BASE_URL = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

BCB_URL = (
    f"{BCB_BASE_URL}"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

print(BCB_URL)

https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-12-2026'&@dataFinalCotacao='09-19-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json


In [0]:
# Requisição à API
import requests

response = requests.get(
    BCB_URL,
    timeout=30
)

response.raise_for_status()

dados_api = response.json()

print(f"Status HTTP: {response.status_code}")
print(f"Quantidade de cotações: {len(dados_api.get('value', []))}")

Status HTTP: 200
Quantidade de cotações: 5


In [0]:
# Extração dos dados da API do Banco Central
cotacoes = dados_api.get("value", [])

if not cotacoes:
    raise ValueError(
        "A API do Banco Central não retornou cotações "
        "para o período informado."
    )

cotacoes[:5]

[{'cotacaoCompra': 5.169, 'dataHoraCotacao': '2026-09-14 13:10:08.144425'},
 {'cotacaoCompra': 5.1484, 'dataHoraCotacao': '2026-09-15 13:09:19.199664'},
 {'cotacaoCompra': 5.152, 'dataHoraCotacao': '2026-09-16 13:05:30.35873'},
 {'cotacaoCompra': 5.1515, 'dataHoraCotacao': '2026-09-17 13:03:21.858212'},
 {'cotacaoCompra': 5.1569, 'dataHoraCotacao': '2026-09-18 13:03:34.742036'}]

In [0]:
# Criação do DataFrame: JSON -> Spark DataFrame
df_cotacao = spark.createDataFrame(cotacoes)

display(df_cotacao)

cotacaoCompra,dataHoraCotacao
5.169,2026-09-14 13:10:08.144425
5.1484,2026-09-15 13:09:19.199664
5.152,2026-09-16 13:05:30.35873
5.1515,2026-09-17 13:03:21.858212
5.1569,2026-09-18 13:03:34.742036


In [0]:
# Adição da coluna ingestion_datetime
df_cotacao_bronze = (
    df_cotacao
    .withColumn(
        "ingestion_datetime",
        current_timestamp()
    )
)

PERSISTÊNCIA DA COTAÇÃO NA BRONZE

In [0]:
# Salvar cotações na tabela
COTACAO_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.tb_cotacao_dolar"

api_count = df_cotacao_bronze.count()

print(f"Registros retornados pela API: {api_count}")

if spark.catalog.tableExists(COTACAO_TABLE):

    bronze_count = spark.table(COTACAO_TABLE).count()

    print(f"Registros existentes na Bronze: {bronze_count}")

    if bronze_count == api_count:
        print(
            "[IGNORADO] A cotação deste período aparentemente "
            "já foi carregada durante o desenvolvimento."
        )

    else:
        print(
            "[DESENVOLVIMENTO] Recriando tb_cotacao_dolar "
            "para evitar duplicações causadas pela reexecução."
        )

        spark.sql(
            f"DROP TABLE IF EXISTS {COTACAO_TABLE}"
        )

        (
            df_cotacao_bronze.write
            .format("delta")
            .mode("append")
            .saveAsTable(COTACAO_TABLE)
        )

        print("[OK] Tabela recriada.")

else:

    (
        df_cotacao_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(COTACAO_TABLE)
    )

    print("[OK] Tabela criada.")

Registros retornados pela API: 5
[OK] Tabela criada.


In [0]:
# Validação final da cotação
df_cotacao_validacao = spark.table(COTACAO_TABLE)

print("VALIDAÇÃO — COTAÇÃO DO DÓLAR")

print(
    f"Quantidade de registros: "
    f"{df_cotacao_validacao.count()}"
)

required_columns = {
    "dataHoraCotacao",
    "cotacaoCompra",
    "ingestion_datetime"
}

existing_columns = set(df_cotacao_validacao.columns)

missing_columns = required_columns - existing_columns

if not missing_columns:
    print("[OK] Todas as colunas obrigatórias estão presentes.")
else:
    print(
        f"[ERRO] Colunas ausentes: {missing_columns}"
    )

display(df_cotacao_validacao)

VALIDAÇÃO — COTAÇÃO DO DÓLAR
Quantidade de registros: 5
[OK] Todas as colunas obrigatórias estão presentes.


cotacaoCompra,dataHoraCotacao,ingestion_datetime
5.169,2026-09-14 13:10:08.144425,2026-09-19T20:37:33.810Z
5.1484,2026-09-15 13:09:19.199664,2026-09-19T20:37:33.810Z
5.152,2026-09-16 13:05:30.35873,2026-09-19T20:37:33.810Z
5.1515,2026-09-17 13:03:21.858212,2026-09-19T20:37:33.810Z
5.1569,2026-09-18 13:03:34.742036,2026-09-19T20:37:33.810Z
